We will work on thse core eval ideas: 
correctness
→ does generated answer match the expected/ground-truth answer?

faithfulness
→ is generated answer supported by retrieved context?

relevancy
→ are retrieved documents/chunks relevant to the question?

# Agenda:

1. Small eval dataset
2. Local RAG function
3. Run questions through RAG
4. Store answer + retrieved chunks
5. Manually score:
   - retrieval relevance
   - answer correctness
   - faithfulness

# Evaluation Basics

Goal: learn how to measure whether a RAG system works.

We will evaluate:
1. Retrieval relevance: did retrieved chunks contain the answer?
2. Correctness: did the generated answer match the expected answer?
3. Faithfulness: was the answer supported by retrieved context?

This local version avoids OpenAI evaluators. We start with manual scoring.

This is phase-3 right now ::::::
manual eval
→ local LLM-as-judge with Ollama
→ automated eval frameworks like DeepEval/Ragas
→ production eval dashboards/regression tests

it starts with mannual EVALS and then we can proceed in this fashion

In [1]:
from pathlib import Path
import ollama

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS

MODEL_NAME="gemma4:e4b"

In [2]:
pdf_path = Path("../data/Understanding_Climate_Change.pdf")

print(pdf_path)

loader = PyPDFLoader(pdf_path)   # here we initialized the loader
documents = loader.load()    # here we are using that loader 

print("PDF exists:", pdf_path.exists())
print("Number of page documents:", len(documents))
print("First page preview:")
print(documents[0].page_content[:300])

../data/Understanding_Climate_Change.pdf
PDF exists: True
Number of page documents: 33
First page preview:
Understanding Climate Change 
Chapter 1: Introduction to Climate Change 
Climate change refers to significant, long-term changes in the global climate. The term 
"global climate" encompasses the planet's overall weather patterns, including temperature, 
precipitation, and wind patterns, over an exte


In [4]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
# we are gonna make chunks from the loaded documents and then turn them into embeddings and the store them in to vector store

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

chunks = splitter.split_documents(documents)

# now we need to convert it into embeddings so we need a model to convert it into embeddings


vector_store = FAISS.from_documents(chunks, embedding_model)

print("Number of chunks:", len(chunks))
print("Vector store type:", type(vector_store))
# print([ele for ele in dir(FAISS) if not ele.startswith("__")])
# print(help(FAISS.from_documents))

Number of chunks: 97
Vector store type: <class 'langchain_community.vectorstores.faiss.FAISS'>
['_FAISS__add', '_FAISS__from', '_abc_impl', '_aembed_documents', '_aembed_query', '_asimilarity_search_with_relevance_scores', '_cosine_relevance_score_fn', '_create_filter_func', '_embed_documents', '_embed_query', '_euclidean_relevance_score_fn', '_get_retriever_tags', '_max_inner_product_relevance_score_fn', '_select_relevance_score_fn', '_similarity_search_with_relevance_scores', 'aadd_documents', 'aadd_texts', 'add_documents', 'add_embeddings', 'add_texts', 'adelete', 'afrom_documents', 'afrom_embeddings', 'afrom_texts', 'aget_by_ids', 'amax_marginal_relevance_search', 'amax_marginal_relevance_search_by_vector', 'amax_marginal_relevance_search_with_score_by_vector', 'as_retriever', 'asearch', 'asimilarity_search', 'asimilarity_search_by_vector', 'asimilarity_search_with_relevance_scores', 'asimilarity_search_with_score', 'asimilarity_search_with_score_by_vector', 'delete', 'deserialize_

Now we are gonna be creating a small EVAL dataset

In [13]:
eval_dataset = [
    {
        "question": "What is the main cause of recent climate change?",
        "expected_answer": "The increase in greenhouse gases in the atmosphere, mainly from human activities such as burning fossil fuels and deforestation.",
    },
    {
        "question": "How does burning fossil fuels contribute to climate change?",
        "expected_answer": "Burning fossil fuels releases large amounts of carbon dioxide, increasing greenhouse gases and warming the climate.",
    },
    {
        "question": "What are some effects of climate change?",
        "expected_answer": "Effects include rising temperatures, melting ice, sea-level rise, extreme weather, droughts, flooding, and ocean acidification.",
    },
    {
        "question": "How does renewable energy help mitigate climate change?",
        "expected_answer": "Renewable energy sources produce little to no greenhouse gas emissions and reduce reliance on fossil fuels.",
    },
    {
        "question": "What is ocean acidification?",
        "expected_answer": "Ocean acidification is the increase in ocean acidity caused by higher CO2 levels dissolving into seawater.",
    },
]

print("Number of eval questions:", len(eval_dataset))
print(eval_dataset[0])

Number of eval questions: 5
{'question': 'What is the main cause of recent climate change?', 'expected_answer': 'The increase in greenhouse gases in the atmosphere, mainly from human activities such as burning fossil fuels and deforestation.'}


In [19]:
from urllib import response


def run_rag(question, k=3):
    retrieved_docs = vector_store.similarity_search(question, k=3)

    context = "\n\n".join(doc.page_content for doc in retrieved_docs)

    prompt = f""" 
        Answer the question using only the context below.
        If the answer is not in the context, say - "I don't know the answer based on the context"
        Question: {question}
        Context: {context}
        Answer in one short sentence
    """

    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    # print(response)

    return {
        "question": question,
        "answer": response.message.content,
        "context": context,
        "retrieved_docs": retrieved_docs,
    }


In [20]:
# now we will test it 
sample = run_rag(eval_dataset[0]["question"], k=3)

print("Question:")
print(sample["question"])

print("\nAnswer:")
print(sample["answer"])

print("\nRetrieved sources:")
for i, doc in enumerate(sample["retrieved_docs"], start=1):
    print(f"\n--- Source {i} ---")
    print("Page:", doc.metadata.get("page_label") or doc.metadata.get("page"))
    print(doc.page_content[:400])


Question:
What is the main cause of recent climate change?

Answer:
The primary cause of recent climate change is the increase in greenhouse gases in the atmosphere.

Retrieved sources:

--- Source 1 ---
Page: 1
Most of these climate changes are attributed to very small variations in Earth's orbit that 
change the amount of solar energy our planet receives. During the Holocene epoch, which 
began at the end of the last ice age, human societies flourished, but the industrial era has seen 
unprecedented changes. 
Modern Observations 
Modern scientific observations indicate a rapid increase in global tempera

--- Source 2 ---
Page: 1
Chapter 2: Causes of Climate Change 
Greenhouse Gases 
The primary cause of recent climate change is the increase in greenhouse gases in the 
atmosphere. Greenhouse gases, such as carbon dioxide (CO2), methane (CH4), and nitrous 
oxide (N2O), trap heat from the sun, creating a "greenhouse effect." This effect is essential 
for life on Earth, as it keeps the p

In [21]:
eval_results = []

for item in eval_dataset:
    question = item["question"]
    expected_answer = item["expected_answer"]

    rag_result = run_rag(question, k=3)

    eval_results.append({
        "question": question,
        "expected_answer": expected_answer,
        "generated_answer": rag_result["answer"],
        "context": rag_result["context"],
        "retrieved_docs": rag_result["retrieved_docs"],
        "retrieval_score": None,
        "correctness_score": None,
        "faithfulness_score": None,
    })

print("Eval results:", len(eval_results))

Eval results: 5


In [22]:
result = eval_results[0]

print("Question:")
print(result["question"])

print("\nExpected answer:")
print(result["expected_answer"])

print("\nGenerated answer:")
print(result["generated_answer"])

print("\nRetrieved context preview:")
print(result["context"][:1000])


Question:
What is the main cause of recent climate change?

Expected answer:
The increase in greenhouse gases in the atmosphere, mainly from human activities such as burning fossil fuels and deforestation.

Generated answer:
The primary cause of recent climate change is the increase in greenhouse gases in the atmosphere.

Retrieved context preview:
Most of these climate changes are attributed to very small variations in Earth's orbit that 
change the amount of solar energy our planet receives. During the Holocene epoch, which 
began at the end of the last ice age, human societies flourished, but the industrial era has seen 
unprecedented changes. 
Modern Observations 
Modern scientific observations indicate a rapid increase in global temperatures, sea levels, 
and extreme weather events. The Intergovernmental Panel on Climate Change (IPCC) has 
documented these changes extensively. Ice core samples, tree rings, and ocean sediments 
provide a historical record that scientists use to und

retrieval_score:
0 = retrieved chunks do not contain the answer
1 = partially contain the answer
2 = clearly contain the answer

correctness_score:
0 = generated answer is wrong
1 = partially correct
2 = correct

faithfulness_score:
0 = answer is not supported by retrieved context
1 = partially supported
2 = clearly supported

In [23]:
# THIS IS DONE MANNUALLY

eval_results[0]["retrieval_score"] = 2
eval_results[0]["correctness_score"] = 2
eval_results[0]["faithfulness_score"] = 2

In [25]:
# A function to view the eval results

def review_eval_result(index):
    result = eval_results[index]

    print("=" * 80)
    print("Index:", index)

    print("\nQuestion:")
    print(result["question"])

    print("\nExpected answer:")
    print(result["expected_answer"])

    print("\nGenerated answer:")
    print(result["generated_answer"])

    print("\nRetrieved chunks:")
    for i, doc in enumerate(result["retrieved_docs"], start=1):
        print(f"\n--- Chunk {i} ---")
        print("Page:", doc.metadata.get("page_label") or doc.metadata.get("page"))
        print(doc.page_content[:700])

    print("\nScores:")
    print("retrieval_score:", result["retrieval_score"])
    print("correctness_score:", result["correctness_score"])
    print("faithfulness_score:", result["faithfulness_score"])

In [ ]:
review_eval_result(0)

retrieval_score
→ did chunks contain the answer?

correctness_score
→ did answer match expected answer?

faithfulness_score
→ is answer supported by retrieved chunks?


AND now we are going to compute averages

In [28]:
def average_score(results, score_name):
    scored_values = [
        result[score_name]
        for result in results
        if result[score_name] is not None
    ]

    if not scored_values:
        return None

    return sum(scored_values) / len(scored_values)


avg_retrieval = average_score(eval_results, "retrieval_score")
avg_correctness = average_score(eval_results, "correctness_score")
avg_faithfulness = average_score(eval_results, "faithfulness_score")

print("Average retrieval score:", avg_retrieval)
print("Average correctness score:", avg_correctness)
print("Average faithfulness score:", avg_faithfulness)


print("Retrieval %:", (avg_retrieval / 2) * 100)
print("Correctness %:", (avg_correctness / 2) * 100)
print("Faithfulness %:", (avg_faithfulness / 2) * 100)

print(eval_results)

Average retrieval score: 2.0
Average correctness score: 2.0
Average faithfulness score: 2.0
Retrieval %: 100.0
Correctness %: 100.0
Faithfulness %: 100.0
[{'question': 'What is the main cause of recent climate change?', 'expected_answer': 'The increase in greenhouse gases in the atmosphere, mainly from human activities such as burning fossil fuels and deforestation.', 'generated_answer': 'The primary cause of recent climate change is the increase in greenhouse gases in the atmosphere.', 'context': 'Most of these climate changes are attributed to very small variations in Earth\'s orbit that \nchange the amount of solar energy our planet receives. During the Holocene epoch, which \nbegan at the end of the last ice age, human societies flourished, but the industrial era has seen \nunprecedented changes. \nModern Observations \nModern scientific observations indicate a rapid increase in global temperatures, sea levels, \nand extreme weather events. The Intergovernmental Panel on Climate Ch

# MANNUAL EVAL is done 

# Now moving towards local LLM as-a-Judge